# HIT401 Simple Scam Email Project — Part 3

## Goal
Compare persuasion cues in **caught scams** and **missed scams** from the external test.

This part answers: **What language or persuasion cues appear in scams that the baseline model misses?**

The cue detector uses transparent keyword rules. It identifies phrases in the messages; it does not measure real users' thoughts or prove that a phrase caused someone to be deceived.

In [ ]:
# Cell 1: Import the tools
from pathlib import Path
import zipfile
import re
import itertools
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 140)
print('Tools loaded successfully.')

## Step 1 — Upload the Part 2 results
Upload `hit401_part2_results.zip`. We use the individual external-test predictions, not manually copied summary numbers.

In [ ]:
# Cell 2: Upload and open the prediction results
zip_path = Path('/content/hit401_part2_results.zip')
if not zip_path.exists():
    from google.colab import files
    print('Choose hit401_part2_results.zip from your computer.')
    uploaded = files.upload()
    if 'hit401_part2_results.zip' not in uploaded:
        raise FileNotFoundError('Please upload hit401_part2_results.zip.')

data_dir = Path('/content/hit401_part3_data')
with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(data_dir)

predictions = pd.read_csv(data_dir / 'external_predictions.csv')
scams = predictions.loc[predictions['label'] == 1].copy().reset_index(drop=True)

print('All external predictions:', len(predictions))
print('External scams used in this analysis:', len(scams))
print(scams['result_type'].value_counts())

## Step 2 — Define eight persuasion-cue dictionaries
Each category has words or phrases selected because they represent a clear idea. The rules are deliberately small so the team can read, question and improve them.

In [ ]:
# Cell 3: Keyword dictionaries
cue_keywords = {
    'urgency': [
        'urgent', 'urgently', 'immediately', 'act now', 'respond now', 'reply now',
        'as soon as possible', 'asap', 'within 24 hours', 'deadline', 'final notice', 'without delay'
    ],
    'authority': [
        'government', 'official', 'department', 'director', 'manager', 'president',
        'bank officer', 'administrator', 'security team', 'help desk', 'police', 'customs'
    ],
    'fear_threat': [
        'suspended', 'closed', 'terminated', 'blocked', 'deactivated', 'penalty',
        'warning', 'legal action', 'unauthorised', 'unauthorized', 'compromised', 'risk'
    ],
    'reward_gain': [
        'prize', 'winner', 'won', 'reward', 'bonus', 'profit', 'inheritance',
        'million dollars', 'lottery', 'beneficiary', 'investment opportunity'
    ],
    'trust_credibility': [
        'confidential', 'legitimate', 'secure', 'verified', 'trusted', 'guarantee',
        'assure', 'faithfully', 'sincerely', 'reliable', 'authentic'
    ],
    'financial_request': [
        'bank account', 'bank details', 'wire transfer', 'transfer funds', 'payment',
        'processing fee', 'transaction', 'deposit', 'credit card', 'account number', 'money'
    ],
    'credential_request': [
        'password', 'login', 'log in', 'sign in', 'verify your account', 'confirm your account',
        'update your account', 'security code', 'pin number', 'username', 'click here'
    ],
    'relationship_emotion': [
        'dear friend', 'dearest', 'beloved', 'love', 'family', 'widow', 'orphan',
        'trust you', 'help me', 'kindly assist', 'god bless', 'in christ', 'sympathy'
    ],
}

print('Cue categories:', list(cue_keywords))

## Step 3 — Tag each scam email
A cue receives `1` when at least one phrase from its dictionary appears, otherwise `0`. We also save the exact matched phrases so the result can be checked manually.

In [ ]:
# Cell 4: Build safe phrase patterns and tag the messages
def phrase_pattern(phrases):
    parts = [r'\b' + re.escape(p.lower()).replace(r'\ ', r'\s+') + r'\b' for p in phrases]
    return re.compile('|'.join(parts), flags=re.IGNORECASE)

patterns = {cue: phrase_pattern(words) for cue, words in cue_keywords.items()}
cue_names = list(cue_keywords)

for cue, pattern in patterns.items():
    scams[cue] = scams['text'].fillna('').str.contains(pattern).astype(int)
    scams[f'{cue}_matches'] = scams['text'].fillna('').apply(
        lambda text: ' | '.join(sorted(set(match.group(0).lower() for match in pattern.finditer(text))))
    )

scams['number_of_cues'] = scams[cue_names].sum(axis=1)
display(scams[['result_type', 'number_of_cues'] + cue_names].head())

## Step 4 — Compare caught and missed scams
We use percentages because the caught and missed groups have different numbers of messages. `Difference` is missed percentage minus caught percentage. A positive value means the cue is more common among missed scams.

In [ ]:
# Cell 5: Cue percentages by model outcome
comparison = (
    scams.groupby('result_type')[cue_names]
    .mean()
    .mul(100)
    .T
)
comparison = comparison[['caught_scam', 'missed_scam']]
comparison.columns = ['Caught scams (%)', 'Missed scams (%)']
comparison['Difference (percentage points)'] = (
    comparison['Missed scams (%)'] - comparison['Caught scams (%)']
)
comparison = comparison.sort_values('Difference (percentage points)', ascending=False)
comparison.round(2)

In [ ]:
# Cell 6: Chart the caught-versus-missed percentages
comparison[['Caught scams (%)', 'Missed scams (%)']].plot(
    kind='barh', figsize=(10, 6), color=['#4C78A8', '#E45756']
)
plt.title('Persuasion cues in caught and missed scams')
plt.xlabel('Messages containing cue (%)')
plt.ylabel('Cue')
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

## Step 5 — Compare the number of cues
This asks whether caught scams contain more of our recognised cues overall. It does not measure psychological strength.

In [ ]:
# Cell 7: Average cue count
cue_count_summary = scams.groupby('result_type')['number_of_cues'].agg(['count', 'mean', 'median'])
cue_count_summary.loc[['caught_scam', 'missed_scam']].round(2)

## Step 6 — Find common cue combinations
A combination means two dictionary categories appeared somewhere in the same scam message.

In [ ]:
# Cell 8: Top pairs for caught and missed scams
pair_rows = []
for outcome in ['caught_scam', 'missed_scam']:
    group = scams.loc[scams['result_type'] == outcome]
    for cue_1, cue_2 in itertools.combinations(cue_names, 2):
        count = int(((group[cue_1] == 1) & (group[cue_2] == 1)).sum())
        percent = 100 * count / len(group)
        pair_rows.append({
            'result_type': outcome, 'cue_1': cue_1, 'cue_2': cue_2,
            'count': count, 'percent': percent
        })

pair_results = pd.DataFrame(pair_rows)
top_pairs = (
    pair_results.sort_values(['result_type', 'percent'], ascending=[True, False])
    .groupby('result_type')
    .head(10)
)
top_pairs.round(2)

## Step 7 — Create a manual-review sample
Keyword rules can make mistakes. Two group members should independently review the same sample and record whether each tag is correct. This is a quality check, not a study of all messages.

In [ ]:
# Cell 9: Random sample of 10 caught and 10 missed scams
review_columns = ['result_type', 'source', 'text', 'scam_probability'] + cue_names
manual_review = pd.concat([
    scams.loc[scams['result_type'] == 'caught_scam', review_columns].sample(10, random_state=42),
    scams.loc[scams['result_type'] == 'missed_scam', review_columns].sample(10, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

manual_review.insert(0, 'review_id', range(1, len(manual_review) + 1))
manual_review['reviewer_1_notes'] = ''
manual_review['reviewer_2_notes'] = ''
manual_review['agreed_tag_corrections'] = ''
display(manual_review[['review_id', 'result_type', 'text']].head())

## Step 8 — Show examples with matched phrases
Use these examples in discussion only after reading the full message and confirming that the matched phrase makes sense in context.

In [ ]:
# Cell 10: Display short, checkable examples
match_columns = [f'{cue}_matches' for cue in cue_names]
example_columns = ['result_type', 'text', 'number_of_cues'] + match_columns
examples = pd.concat([
    scams.loc[scams['result_type'] == 'caught_scam', example_columns].sample(3, random_state=7),
    scams.loc[scams['result_type'] == 'missed_scam', example_columns].sample(3, random_state=7),
]).reset_index(drop=True)
display(examples)

## Step 9 — Save the analysis

In [ ]:
# Cell 11: Export the tables
output_dir = Path('/content/hit401_part3_results')
output_dir.mkdir(exist_ok=True)
comparison.to_csv(output_dir / 'caught_vs_missed_cue_percentages.csv')
cue_count_summary.to_csv(output_dir / 'cue_count_summary.csv')
pair_results.to_csv(output_dir / 'cue_pair_results.csv', index=False)
manual_review.to_csv(output_dir / 'manual_review_sample.csv', index=False)
scams.to_csv(output_dir / 'scams_with_cue_tags.csv', index=False)

archive_path = Path('/content/hit401_part3_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_dir.glob('*.csv'):
        archive.write(file_path, arcname=file_path.name)

print('Saved:', archive_path)
print('Download this ZIP from the Colab Files panel.')

## What to tell the supervisor

> We compared the external scams that Logistic Regression caught with those it missed. We applied eight transparent keyword dictionaries covering cues such as urgency, authority, fear, reward, financial requests and credential requests. We compared percentages rather than raw counts because the two outcome groups were different sizes. We also generated a manual-review sample so two team members can check whether the automated tags are sensible.

## Limitations
- A keyword match can be correct, irrelevant or ambiguous depending on context.
- The dictionaries do not capture every possible wording.
- The tags describe message content, not actual user behaviour or susceptibility.
- Differences may reflect dataset source and writing style as well as model weakness.